# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import boto3
import s3fs
import matplotlib.pyplot as plt
import seaborn as sns
# Weather data library
from meteostat import Stations, Daily
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

# Connect SageMaker to S3

In [ ]:
#Create S3 client using boto3
#This allows the notebook to communicate with S3
s3 = boto3.client('s3')

#Define your S3 bucket name where your datasets are stored
bucket = "vegetation-risk-ml"

#list files inside bucket
response = s3.list_objects_v2(Bucket=bucket)

#Print the names of the files in the bucket
for obj in response['Contents']:
    print(obj['Key'])

The datasets used in this project were originally downloaded from public government data sources- the U.S. Forest Service (FIA), CAL FIRE wildfire records and NOAA weather data accessed through the Meteostat library. After downloading, the files were manually uploaded to an Amazon S3 bucket. The datasets are accessed from Amazon S3 using SageMaker Studio notebooks for exploration and further processing.

# Load FIA(forest vegetation data) from S3

The FIA forest vegetation dataset consists of three tables: subplot information, tree characteristics, and biomass measurements. These files are stored in Amazon S3 and loaded into the SageMaker notebook for further analysis. 

In [ ]:
#file path for FIA forest datasets inside S3 bucket
subplot_path = "s3://vegetation-risk-ml/raw/forest/CA_SUBPLOT.csv"
tree_path = "s3://vegetation-risk-ml/raw/forest/CA_TREE.csv"
biomass_path = "s3://vegetation-risk-ml/raw/forest/CA_TREE_REGIONAL_BIOMASS.csv"

In [ ]:
# Load the CSV file from S3 into a pandas DataFrame
subplot = pd.read_csv(subplot_path, low_memory=False)
subplot.head(3)

In [ ]:
# Load the tree dataset from the specified file path into a pandas DataFrame.
trees = pd.read_csv(tree_path, low_memory=False)
trees.head(3)

In [ ]:
# Load the biomass dataset from the specified file path into a pandas DataFrame
biomass = pd.read_csv(biomass_path, low_memory=False)
biomass.head(3)

In [ ]:
#check data structure of subplot dataset
subplot.info()

In [ ]:
#check data structure of tree dataset
trees.info()

In [ ]:
#check data structure of biomass dataset
biomass.info()

#### Merge tree,biomass and subplot dataset

In [ ]:
#Merge trees and subplot.To save memory, we will choose needed columns only
combined_forest = trees[["CN", "PLT_CN", "DIA", "HT", "SPCD", "STATUSCD"]].merge(subplot[["PLT_CN", "SLOPE", "ASPECT"]],on="PLT_CN",how="left")

#Merge the resulting dataframe with biomass dataframe on tree identifier.Again, select needed columns to save memory
combined_forest = combined_forest.merge(biomass[["TRE_CN", "REGIONAL_DRYBIOT", "REGIONAL_DRYBIOM"]],left_on="CN",right_on="TRE_CN",how="left")
combined_forest = combined_forest.drop_duplicates()

#Check the resulting dataframe
print("Merged forest dataset shape:", combined_forest.shape)
combined_forest.head()

The three FIA datasets are merged to create a unified forest vegetation dataset. The merge connects tree measurements with subplot environmental characteristics and biomass estimates.

In [ ]:
#Save this merged dataframe to S3 for future use
combined_forest.to_csv("s3://vegetation-risk-ml/processed/forest/combined_forest_data.csv", index=False)

# Load CALFIRE dataset from S3

The CAL FIRE dataset contains wildfire perimeter records for California, originaly downloaded from public website and uploaded manully into S3 in raw/fire folder.This dataset provides information on fire location, cause, start date, and burned area.

In [ ]:
# Path to wildfire dataset stored in S3
fire_path = "s3://vegetation-risk-ml/raw/fire/California_Historic_Fire_data.csv"

# Load wildfire dataset
fire = pd.read_csv(fire_path)

# Display dataset shape and rows
print("Fire dataset shape:", fire.shape)
fire.head(3)

In [ ]:
#Basic structure of fire dataset
print(fire.info())

In [ ]:
#missing values in fire dataset
print(fire.isnull().sum())

In [ ]:
#duplicated values in fire dataset
print(fire.duplicated().sum())

# Load Weather Data Using Meteostat

Weather stations across California are retrieved using the Meteostat API. These stations provide historical weather measurements such as temperature and precipitation.

In [ ]:
# Retrieve weather stations located in California
stations = Stations()

stations = stations.region('US', 'CA')
stations = stations.fetch(100)

print(stations.head())

# download weather data
start = datetime(2015,1,1)
end = datetime(2026,1,1)

weather_data = []

for station_id in stations.index[:50]:
    
    try:
        data = Daily(station_id, start, end).fetch()
        data["station"] = station_id
        weather_data.append(data)
        
    except:
        pass

weather = pd.concat(weather_data)

weather.reset_index(inplace=True)

weather.head()

In [ ]:
#Summary statistics of weather data
weather.describe()

In [ ]:
#missing values
print(weather.isnull().sum())

In [ ]:
# save the weather data to S3 for future use
weather.to_csv("s3://vegetation-risk-ml/raw/weather/california_weather_data.csv", index=False)

# DATA EXPLORATION

### What are the distributions and relationships among key weather variables (temperature, precipitation, and wind speed) across California?
    

In [ ]:

weather_subset = weather[["tavg","prcp","wspd"]]

sns.pairplot(weather_subset.sample(2000),diag_kind="kde", height=2)

plt.suptitle("Distribution and Relationships of Weather Variables", y=1.02)
plt.show()

The pairplot shows the distribution and relationships between temperature, precipitation, and wind speed across California weather stations. Temperature appears approximately normally distributed, while precipitation is heavily right-skewed, indicating that most days have little or no rainfall with occasional heavy precipitation events. Wind speed shows moderate variability but is generally concentrated around lower values. The scatter relationships suggest weak correlations between these variables, indicating that temperature, precipitation, and wind speed vary relatively independently across the dataset.

### Is wildfire size distribution skewed toward extreme events?

In [ ]:
import numpy as np

plt.figure(figsize=(6,4))
sns.histplot(np.log1p(fire["incident_acres_burned"]),kde=True, bins=50)

plt.xlabel("Log Wildfire Size (Acres)")
plt.title("Distribution of Wildfire Size")
plt.show()

Plot shows Wildfire data is right-skewed.Most fires burn small areas, while a small number of large fires account for a significant portion of total burned land. This pattern is common in wildfire datasets and highlights the importance of identifying conditions that lead to extreme fire events.

### What is the distribution of vegetation biomass across forest plots?

This directly uses your forest dataset and helps understand fuel load, which is important for wildfire risk.

In [ ]:
sns.boxplot(x=np.log1p(combined_forest["REGIONAL_DRYBIOT"].dropna()))

plt.title("Vegetation Biomass Distribution")
plt.show()

The vegetation biomass distribution is highly right-skewed, indicating that most forest plots contain relatively low to moderate biomass while a small number of plots have extremely high biomass values. These high-biomass areas may represent dense vegetation and which can act as larger fuel sources and potentially increase wildfire intensity if ignition occurs.

### Where are wildfire incidents geographically concentrated across California?

In [ ]:
plt.figure(figsize=(7,6))
fire_plt = fire.dropna(subset=["incident_longitude","incident_latitude"])
plt.hexbin(fire_plt["incident_longitude"],fire_plt["incident_latitude"],gridsize=40,cmap="Reds")

plt.colorbar(label="Number of Fires")

plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Wildfire Density Across California")

plt.show()

The hexbin plot highlights the geographic density of wildfire incidents across California. Areas with darker hexagons indicate regions with a higher concentration of wildfire events. These clusters suggest locations where environmental conditions and vegetation fuel loads may contribute to increased wildfire risk. Identifying such hotspots can support vegetation management strategies, including prioritizing tree trimming in high-risk regions.